# Recuperação de Informação: TF-IDF × BM25 no acervo de uma biblioteca

**Projeto 2: Estatística e Probabilidade para Computação (CIn-UFPE, 2026.2)**

**Equipe:** Arthur Dias, Eduardo Henrique, Felipe Augusto, Gabriel André e Philip Santiago.

Repositório: https://github.com/philipsantiagoo/acervo-rank

## 0. Configuração

Instalação e carregamento dos pacotes, e endereço dos dados no GitHub.

In [ ]:
# Pacotes: serão adicionados nas próximas fases

# Endereço dos dados do CISI no repositório do projeto
url_dados <- "https://raw.githubusercontent.com/philipsantiagoo/acervo-rank/main/data/raw/"


## 1. Introdução

### 1.1. Contextualização

Uma biblioteca universitária quer melhorar a busca do seu acervo de artigos. Hoje, o sistema ordena os resultados com **TF-IDF**, um modelo vetorial que dá peso às palavras conforme a frequência delas no documento e a raridade na coleção. A proposta é trocar pelo **BM25**, um modelo probabilístico que é o padrão em motores de busca como Lucene e Elasticsearch.

O BM25 corrige dois defeitos conhecidos do TF-IDF:

- **saturação:** repetir uma palavra muitas vezes não aumenta a nota sem limite;
- **normalização pelo tamanho:** documentos longos não ganham pontos só por serem longos.

Trocar de sistema tem custo. Por isso, a biblioteca só quer fazer a troca se houver **evidência estatística** de que os usuários seriam mais bem atendidos. Este projeto dá continuidade ao plano apresentado no Projeto 1.


### 1.2. Objetivo

**Pergunta central:** com o BM25, os pesquisadores encontram os artigos certos mais perto do topo da lista do que com o TF-IDF, e essa melhora é estatisticamente significativa?

Para responder, vamos:

1. explorar a coleção e testar se o **tamanho dos artigos** está relacionado à relevância, o que justificaria a normalização feita pelo BM25 (seção 3);
2. implementar os dois modelos e ordenar todos os artigos para cada consulta (seção 4);
3. medir a qualidade de cada ranking com **P@10, MAP e nDCG@10** e comparar os modelos consulta a consulta com um **teste t pareado unilateral** (α = 0,05) (seção 5).

**Hipótese (Projeto 1):** com o BM25, os pesquisadores verão mais artigos relevantes no topo da busca do que com o TF-IDF, porque ele não supervaloriza termos repetidos nem documentos longos.


### 1.3. Descrição dos Dados

Um acervo real não tem gabarito: não se sabe quais artigos respondem cada pergunta. Por isso, a coleção **CISI** faz o papel do acervo da biblioteca. É uma coleção de teste clássica de Recuperação de Informação, distribuída pela Universidade de Glasgow (https://ir.dcs.gla.ac.uk/resources/test_collections/cisi/), com resumos de artigos de ciência da informação, consultas de pesquisadores e julgamentos de relevância feitos por pessoas.

São três arquivos de texto, salvos no repositório do projeto:

| Arquivo | Conteúdo |
| --- | --- |
| `CISI.ALL` | 1.460 artigos: título, autores, resumo e referências cruzadas |
| `CISI.QRY` | 112 consultas |
| `CISI.REL` | Gabarito: pares (consulta, artigo relevante) |

Nos arquivos `.ALL` e `.QRY`, cada campo começa com um marcador no início da linha (`.I` identificador, `.T` título, `.A` autores, `.W` texto, `.X` referências, `.B` ano ou fonte) e pode ocupar várias linhas.

**Dicionário de dados** (tabelas montadas na seção 2.1):

**`artigos`**: um artigo por linha.

| Coluna | Tipo | Descrição |
| --- | --- | --- |
| `doc_id` | inteiro | Identificador do artigo (1 a 1.460) |
| `titulo` | texto | Título do artigo (campo `.T`) |
| `resumo` | texto | Resumo do artigo (campo `.W`) |
| `texto` | texto | Título + resumo: o texto que os modelos vão usar |

**`consultas`**: uma consulta por linha.

| Coluna | Tipo | Descrição |
| --- | --- | --- |
| `query_id` | inteiro | Identificador da consulta (1 a 112) |
| `texto` | texto | Texto da consulta (campo `.W`) |
| `tipo` | texto | `"pergunta"` (escrita por um pesquisador) ou `"artigo"` (resumo de um artigo usado como consulta) |
| `tem_gabarito` | lógico | Se a consulta tem pelo menos um artigo relevante no gabarito |

**`gabarito`**: um par relevante por linha.

| Coluna | Tipo | Descrição |
| --- | --- | --- |
| `query_id` | inteiro | Identificador da consulta |
| `doc_id` | inteiro | Identificador de um artigo relevante para essa consulta |

A relevância é **binária**: se um par (consulta, artigo) não está no gabarito, o artigo não é relevante para a consulta.


## 2. Análise Exploratória de Dados (EDA)

### 2.1. Carregamento e Limpeza dos Dados

Os arquivos são lidos direto do GitHub, para o notebook funcionar em qualquer máquina. Como o formato do CISI é baseado em marcadores, e não em colunas, escrevemos funções próprias de leitura.

Ao inspecionar os arquivos, encontramos alguns detalhes que a leitura precisa tratar:

- **Quebra de linha do Windows:** todas as linhas terminam com `\r\n`. O `\r` é removido.
- **Marcadores não documentados:** além de `.T`, `.A`, `.W` e `.X`, o `CISI.ALL` tem `.B` (ano de publicação, em 24 artigos), `.K` (palavras-chave) e `.C` (códigos de classificação), esses dois só no artigo 321. Por isso, **qualquer** marcador encerra o campo anterior. Se só os conhecidos fossem reconhecidos, as palavras-chave do artigo 321 entrariam no resumo dele.
- **Espaço depois do marcador:** em 15 linhas, o marcador aparece como `.T ` em vez de `.T`.
- **Linhas em branco** no meio dos textos (43 no `CISI.ALL`), que são descartadas.
- **Campos em várias linhas:** as linhas de um mesmo campo são unidas com espaço, e espaços repetidos são reduzidos a um.


In [ ]:
# Lê um arquivo do CISI direto do GitHub.
# Entrada: nome do arquivo (ex.: "CISI.ALL"). Saída: vetor de linhas, sem o "\r" do fim.
ler_linhas <- function(arquivo) {
  con <- url(paste0(url_dados, arquivo))
  linhas <- readLines(con, warn = FALSE, encoding = "UTF-8")
  close(con)
  sub("\r$", "", linhas)
}

# Junta espaços repetidos (inclusive quebras e tabulações) e tira os das pontas.
# Entrada: vetor de textos. Saída: vetor de textos limpos.
limpar <- function(x) trimws(gsub("\\s+", " ", x))

# Separa as linhas de CISI.ALL ou CISI.QRY em registros e campos.
# Entrada: vetor de linhas. Saída: data frame com uma linha por registro, a coluna
# `id` e uma coluna por marcador encontrado (T, A, W, X, ...), com o texto do campo.
ler_registros <- function(linhas) {
  eh_id       <- grepl("^\\.I\\s+[0-9]+\\s*$", linhas)
  eh_marcador <- eh_id | grepl("^\\.[A-Z]\\s*$", linhas)   # aceita espaço depois do marcador
  registro    <- cumsum(eh_id)                              # a qual registro cada linha pertence
  campo       <- ifelse(eh_marcador, substr(linhas, 2, 2), NA)
  ultimo      <- cummax(ifelse(eh_marcador, seq_along(linhas), 0L))
  campo_atual <- c(NA, campo)[ultimo + 1]                   # último marcador visto até a linha
  conteudo    <- !eh_marcador & trimws(linhas) != ""        # descarta marcadores e linhas em branco

  textos <- tapply(linhas[conteudo],
                   list(registro[conteudo], campo_atual[conteudo]),
                   paste, collapse = " ")
  tabela <- as.data.frame(textos, stringsAsFactors = FALSE)
  ids <- as.integer(sub("^\\.I\\s+", "", linhas[eh_id]))
  tabela$id <- ids[as.integer(rownames(textos))]
  rownames(tabela) <- NULL
  tabela
}


**Artigos.** Cada artigo vira uma linha. O texto que os modelos vão usar é **título + resumo**: são os campos que dizem do que o artigo trata. Autores, ano, referências cruzadas e palavras-chave ficam de fora.


In [ ]:
registros_all <- ler_registros(ler_linhas("CISI.ALL"))

artigos <- data.frame(
  doc_id = registros_all$id,
  titulo = limpar(registros_all$T),
  resumo = limpar(registros_all$W)
)
artigos$texto <- paste(artigos$titulo, artigos$resumo)

head(artigos[, c("doc_id", "titulo")])


**Consultas.** Usamos o campo `.W` de cada consulta. Ao ler o arquivo, notamos que existem **dois tipos de consulta**: perguntas escritas por pesquisadores (só `.W`) e resumos de artigos usados como consulta (com `.T`, `.A` e `.B`, além do `.W`). Guardamos o tipo na coluna `tipo` para analisar na EDA.


In [ ]:
registros_qry <- ler_registros(ler_linhas("CISI.QRY"))

consultas <- data.frame(
  query_id = registros_qry$id,
  texto    = limpar(registros_qry$W),
  tipo     = ifelse(is.na(registros_qry$T), "pergunta", "artigo")
)

head(consultas)


**Gabarito.** O `CISI.REL` tem quatro colunas: consulta, artigo relevante e duas colunas sem uso (sempre `0` e `0.000000`), que descartamos.


In [ ]:
gabarito <- read.table(text = ler_linhas("CISI.REL"),
                       col.names = c("query_id", "doc_id", "nao_usada_1", "nao_usada_2"))
gabarito <- gabarito[, c("query_id", "doc_id")]

consultas$tem_gabarito <- consultas$query_id %in% gabarito$query_id

head(gabarito)


**Estrutura das tabelas.** Tipos de cada coluna e resumo numérico do gabarito.


In [ ]:
str(artigos)
str(consultas)
str(gabarito)
summary(gabarito)


**Valores faltantes e duplicados.** Conferimos valores `NA`, textos vazios, identificadores repetidos e pares repetidos no gabarito.


In [ ]:
faltantes <- data.frame(
  tabela = c("artigos", "consultas", "gabarito"),
  valores_NA = c(sum(is.na(artigos)), sum(is.na(consultas)), sum(is.na(gabarito))),
  textos_vazios = c(sum(artigos$titulo == "" | artigos$resumo == ""), sum(consultas$texto == ""), NA),
  ids_duplicados = c(sum(duplicated(artigos$doc_id)), sum(duplicated(consultas$query_id)), NA),
  pares_duplicados = c(NA, NA, sum(duplicated(gabarito)))
)
faltantes


**Conferência.** Comparamos as contagens com os valores esperados da coleção. Se alguma não bater, a célula para com erro.


In [ ]:
conferencia <- data.frame(
  verificacao = c("Artigos", "Consultas", "Consultas com gabarito", "Pares no gabarito",
                  "Consultas do tipo artigo", "Consultas do tipo artigo com gabarito",
                  "Gabarito aponta para artigos existentes",
                  "Palavras-chave do artigo 321 fora do resumo"),
  esperado = c(1460, 112, 76, 3114, 55, 26, TRUE, TRUE),
  obtido = c(nrow(artigos), nrow(consultas), sum(consultas$tem_gabarito), nrow(gabarito),
             sum(consultas$tipo == "artigo"),
             sum(consultas$tipo == "artigo" & consultas$tem_gabarito),
             all(gabarito$doc_id %in% artigos$doc_id),
             !grepl("text searching", artigos$resumo[artigos$doc_id == 321]))
)
conferencia$ok <- conferencia$esperado == conferencia$obtido
conferencia
stopifnot(all(conferencia$ok))


**Consultas com e sem gabarito, por tipo.**


In [ ]:
table(tipo = consultas$tipo, tem_gabarito = consultas$tem_gabarito)


**Resumo da limpeza:**

- Não há valores faltantes, textos vazios nem duplicados nas três tabelas.
- Todos os artigos do gabarito existem na coleção.
- **36 das 112 consultas não têm gabarito** e ficam de fora da avaliação dos modelos, porque sem gabarito não há como medir. Restam **76 consultas**.
- **Desvio em relação ao Projeto 1:** no Projeto 1, descrevemos as consultas como "escritas por pesquisadores". A leitura mostrou que 55 das 112 consultas são, na verdade, resumos de artigos usados como consulta, e 26 delas têm gabarito. Mantemos as 76 consultas com gabarito, usando o campo `.W` de todas, e comparamos os dois tipos na EDA.


### 2.2. Análise Univariada

*A preencher.*

### 2.3. Análise Bivariada e Multivariada

*A preencher.*

### 2.4. Insights da EDA

*A preencher.*

## 3. Validação Estatística (Teste de Hipóteses)

### 3.1. Definição das Hipóteses

*A preencher.*

### 3.2. Escolha e Execução do Teste

*A preencher.*

### 3.3. Interpretação do p-valor

*A preencher.*

## 4. Pré-processamento e Aplicação da Técnica

### 4.1. Preparação dos Dados para o Modelo

*A preencher.*

### 4.2. Modelagem

*A preencher.*

## 5. Resultados e Interpretação

### 5.1. Apresentação dos Resultados

*A preencher.*

### 5.2. Análise Crítica dos Resultados

*A preencher.*

## 6. Conclusão

### 6.1. Resumo dos Insights

*A preencher.*

### 6.2. Recomendações Práticas

*A preencher.*

### 6.3. Limitações e Próximos Passos

*A preencher.*

## Referências

*A preencher.*